In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

from var import DATA_OUT

In [ ]:
MAV_WINDOW = 5
LAG_WINDOW = 180
HORIZON = 10
max_window = max(MAV_WINDOW, LAG_WINDOW)
hrs_, mins_ = divmod(max_window, 60)

df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=['perc_mild_scint', 'perc_strong_scint', 's4_max']
)

# Pre-filtering
df = df[
    (df.index.hour > 17) | (df.index.hour < 6) | (
        (df.index.hour == (17 - hrs_)) & (df.index.minute >= (60 - mins_))
    )
]

# MAVs
df[f"s4_mean_ema_{MAV_WINDOW}m"] = (
    df['s4_mean'].ewm(span=MAV_WINDOW).mean()
)

# Lags
df[f"h_tmk_lag_{LAG_WINDOW}m"] = (
    df['h_tmk'].shift(LAG_WINDOW)
)

# Filtering
df = df[(df.index.hour >= 18) | (df.index.hour < 6)]

# Multi-step target
y_cols = [f's4_mean_{i+1}' for i in range(HORIZON)]
for i in range(HORIZON):
    df[y_cols[i]] = df['s4_mean'].shift(-(i+1))

In [ ]:
TRAIN_START, TRAIN_STOP = '2022-01-01', '2023-01-31'
CALIB_START, CALIB_STOP = '2023-03-01', '2023-03-17'
TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'

In [ ]:
X_cols = [
    'n_sat',
    's4_mean',
    'field_magnitude_avg',
    'wind_speed',
    'wind_density',
    'wind_pressure',
    'eletric_field',
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_ema_5m',
    'h_tmk_lag_180m',
]

X_train, X_calib, X_test = (
    df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(),
    df.loc[CALIB_START:CALIB_STOP, X_cols].copy(),
    df.loc[TEST_START:TEST_STOP, X_cols].copy(),
)
y_train, y_calib, y_test = (
    df.loc[TRAIN_START:TRAIN_STOP, y_cols].copy().fillna(0),
    df.loc[CALIB_START:CALIB_STOP, y_cols].copy().fillna(0),
    df.loc[TEST_START:TEST_STOP, y_cols].copy().fillna(0),
)

In [ ]:
cb_params = {
    'iterations': 500,
    'l2_leaf_reg': 30,
    'max_depth': 4,
    'min_data_in_leaf': 50,
    'thread_count': -1,
    'has_time': True,
    'bootstrap_type': 'Bernoulli',
    'sampling_frequency': 'PerTree',
    'subsample': 0.9,
    'colsample_bylevel': 0.9,
    'verbose': 1,
    'random_seed': 42,
}

In [ ]:
cb_mimo = CatBoostRegressor(
    loss_function='MultiRMSE',
    **cb_params,
)

In [ ]:
cb_mimo.fit(
    pd.concat([X_train, X_calib]),
    pd.concat([y_train, y_calib]),
)

In [ ]:
y_pred = cb_mimo.predict(X_test)

In [ ]:
df_eval = pd.concat(
    [
        y_test,
        pd.DataFrame(
            y_pred, columns=[f's4_mean_pred_{i+1}' for i in range(HORIZON)], index=y_test.index
        ),
    ],
    axis=1,
)

In [ ]:
rmse_vals = []
rrmse_vals = []
steps = y_test.shape[1]

for i in range(steps):
    rmse = root_mean_squared_error(df_eval.iloc[:, i], df_eval.iloc[:, i+steps])
    rrmse = rmse / y_train.iloc[:, i].mean()
    
    rmse_vals.append(rmse)
    rrmse_vals.append(rrmse)

In [ ]:
rmse_vals, rrmse_vals

In [ ]:
k = 0

y_test.iloc[k].values, y_pred[k]

## UQ